# Experiment 1 — Word-context representation

In [1]:
import numpy as np

# 1. Định nghĩa corpus nhỏ
corpus = [
    "the doctor works at the hospital",
    "the physician works at the hospital",
    "the doctor drives a fast car",
    "the physician drives a fast car",
    "the monkey eats a yellow banana",
    "a doctor and a physician cure patients at the hospital"
]

# 2. Tiền xử lý và tạo Vocabulary
# Tách từ và chuyển thành chữ thường
tokenized_corpus = [sentence.lower().split() for sentence in corpus]

# Lấy các từ unique để tạo tập từ vựng V
vocab = set(word for sentence in tokenized_corpus for word in sentence)
vocab = sorted(list(vocab)) # Sắp xếp alphabet cho dễ theo dõi
vocab_size = len(vocab)

# Tạo từ điển map từ <-> index
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}

# 3. Hàm xây dựng Word-Context Matrix
def build_co_occurrence_matrix(tokenized_corpus, vocab_size, word_to_id, window_size):
    # Khởi tạo ma trận X toàn số 0 với kích thước |V| x |V|
    X = np.zeros((vocab_size, vocab_size))
    
    for tokens in tokenized_corpus:
        length = len(tokens)
        for i, target_word in enumerate(tokens):
            target_id = word_to_id[target_word]
            
            # Xác định phạm vi context window
            start = max(0, i - window_size)
            end = min(length, i + window_size + 1)
            
            for j in range(start, end):
                if i != j: # Không tính chính nó
                    context_word = tokens[j]
                    context_id = word_to_id[context_word]
                    X[target_id, context_id] += 1
                    
    return X

# 4. Hàm tính Cosine Similarity
def cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# 5. Thực hiện Experiment với các window = 1, 2, 5
windows = [1, 2, 5]
words_to_compare = [("doctor", "physician"), ("doctor", "hospital")]

print("=== EXPERIMENT 1: WORD-CONTEXT REPRESENTATION ===")
print(f"Vocabulary: {vocab}")
print(f"Vocabulary size (|V|): {vocab_size}")
print(f"Matrix size (|V| x |V|): {vocab_size} x {vocab_size} = {vocab_size * vocab_size} elements\n")

for w in windows:
    # Xây dựng ma trận cho window w
    X = build_co_occurrence_matrix(tokenized_corpus, vocab_size, word_to_id, window_size=w)
    
    # Tính số lượng non-zero
    non_zero_values = np.count_nonzero(X)
    sparsity = 1.0 - (non_zero_values / (vocab_size * vocab_size))
    
    print(f"--- THỬ NGHIỆM VỚI WINDOW = {w} ---")
    print(f"1. Số lượng non-zero values: {non_zero_values} (Độ thưa - Sparsity: {sparsity:.2%})")
    
    # Tính similarity
    print("2. Cosine Similarity:")
    for w1, w2 in words_to_compare:
        id1, id2 = word_to_id[w1], word_to_id[w2]
        sim = cosine_similarity(X[id1], X[id2])
        print(f"   + cos({w1}, {w2}) = {sim:.4f}")
    print("-" * 40)

=== EXPERIMENT 1: WORD-CONTEXT REPRESENTATION ===
Vocabulary: ['a', 'and', 'at', 'banana', 'car', 'cure', 'doctor', 'drives', 'eats', 'fast', 'hospital', 'monkey', 'patients', 'physician', 'the', 'works', 'yellow']
Vocabulary size (|V|): 17
Matrix size (|V| x |V|): 17 x 17 = 289 elements

--- THỬ NGHIỆM VỚI WINDOW = 1 ---
1. Số lượng non-zero values: 48 (Độ thưa - Sparsity: 83.39%)
2. Cosine Similarity:
   + cos(doctor, physician) = 0.8750
   + cos(doctor, hospital) = 0.7071
----------------------------------------
--- THỬ NGHIỆM VỚI WINDOW = 2 ---
1. Số lượng non-zero values: 80 (Độ thưa - Sparsity: 72.32%)
2. Cosine Similarity:
   + cos(doctor, physician) = 0.9075
   + cos(doctor, hospital) = 0.5145
----------------------------------------
--- THỬ NGHIỆM VỚI WINDOW = 5 ---
1. Số lượng non-zero values: 136 (Độ thưa - Sparsity: 52.94%)
2. Cosine Similarity:
   + cos(doctor, physician) = 0.9445
   + cos(doctor, hospital) = 0.6761
----------------------------------------


# Experiment 2 — Word2Vec

In [3]:
from gensim.models import Word2Vec

# 1. Corpus (Tập dữ liệu đầu vào đã được token hóa)
corpus = [
    ["the", "doctor", "works", "at", "the", "hospital"],
    ["the", "physician", "works", "at", "the", "hospital"],
    ["the", "doctor", "drives", "a", "fast", "car"],
    ["the", "monkey", "eats", "a", "yellow", "banana"]
]

# 2. Khởi tạo và huấn luyện mô hình Word2Vec
model = Word2Vec(
    sentences=corpus,        # Corpus
    vector_size=100,         # Embedding dimension: 100 chiều
    window=2,                # Context window: 2 từ trái, 2 từ phải
    min_count=1,             # Min count: Giữ lại tất cả các từ (do corpus quá nhỏ)
    epochs=20,               # Epochs: Lặp 20 lần qua corpus
    sg=1,                    # Training objective: 1 = Skip-gram (0 = CBOW)
    negative=5               # Objective optimization: Negative Sampling với 5 negative samples
)

# 3. Trích xuất Vocabulary được tạo ra
vocabulary = list(model.wv.key_to_index.keys())
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")

# 4. Truy xuất Embedding Vector và tính Similarity
vector_doctor = model.wv['doctor']
print(f"\nKích thước vector của 'doctor': {vector_doctor.shape}") # Output: (100,)

similarity = model.wv.similarity('doctor', 'physician')
print(f"Cosine similarity giữa 'doctor' và 'physician': {similarity:.4f}")

# 5. Tìm các từ gần nhất
similar_words = model.wv.most_similar('doctor', topn=2)
print(f"Top 2 từ gần với 'doctor' nhất: {similar_words}")

Vocabulary size: 14
Vocabulary: ['the', 'a', 'hospital', 'at', 'works', 'doctor', 'banana', 'yellow', 'eats', 'monkey', 'car', 'fast', 'drives', 'physician']

Kích thước vector của 'doctor': (100,)
Cosine similarity giữa 'doctor' và 'physician': 0.0440
Top 2 từ gần với 'doctor' nhất: [('fast', 0.16698160767555237), ('works', 0.13884708285331726)]


# 19. Experiment 3 — Context window

In [4]:
from gensim.models import Word2Vec
import pandas as pd

# 1. Tạo corpus giả lập (các câu đủ dài để thấy sự khác biệt của window=10)
corpus = [
    "the expert doctor works hard every day at the large city hospital to treat the sick patient".split(),
    "the expert physician works hard every day at the large city hospital to treat the sick patient".split(),
    "a good doctor always arrives early at the hospital to check on the patient".split(),
    "a good physician always arrives early at the hospital to check on the patient".split(),
    "the cute little cat eats fresh fish and drinks warm milk in the kitchen".split(),
    "the cute little dog eats fresh meat and drinks cold water in the kitchen".split(),
    "my cat is a very friendly and playful pet to have at home".split(),
    "my dog is a very friendly and playful pet to have at home".split()
]

# 2. Khai báo các tham số
window_sizes = [2, 5, 10]
word_pairs = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]
results = []

print("Đang huấn luyện các mô hình...\n")

# 3. Huấn luyện 3 model với 3 window size khác nhau
for w in window_sizes:
    # Set seed=42 để kết quả ổn định qua các lần chạy, epochs=100 vì corpus nhỏ
    model = Word2Vec(sentences=corpus, vector_size=50, window=w, min_count=1, epochs=100, seed=42, sg=1)
    
    # Lấy similarity cho từng cặp từ
    sims = []
    for w1, w2 in word_pairs:
        sim = model.wv.similarity(w1, w2)
        sims.append(round(sim, 4))
    
    results.append(sims)

# 4. Hiển thị kết quả dưới dạng Bảng (DataFrame)
df = pd.DataFrame(
    results, 
    columns=[f"{w1} - {w2}" for w1, w2 in word_pairs], 
    index=[f"Window {w}" for w in window_sizes]
)

# Transpose (xoay bảng) để giống format đề bài yêu cầu
df_transposed = df.T
df_transposed.columns.name = "Word pair"
print(df_transposed.to_string())

Đang huấn luyện các mô hình...

Word pair           Window 2  Window 5  Window 10
doctor - physician    0.8405    0.9813     0.9910
doctor - hospital     0.9208    0.9904     0.9945
cat - dog             0.8442    0.9715     0.9914


Không, window lớn hơn không phải lúc nào cũng tốt hơn. Khái niệm tốt phụ thuộc hoàn toàn vào mục tiêu của bài toán thực tế.

Dựa vào kết quả bạn vừa chạy, chúng ta có thể thấy rõ tác dụng phụ của việc dùng window quá lớn:

1. Hiện tượng bão hòa ngữ nghĩa
Nhìn vào cột Window 10, độ tương đồng của tất cả các cặp từ đều bị đẩy lên mức cực đại (trên 0.99). Khi kích thước cửa sổ là 10, nó bao trùm gần như toàn bộ chiều dài của một câu trong corpus giả lập. Hậu quả là tất cả các từ trong câu đều trở thành ngữ cảnh của nhau. Mô hình không còn phân biệt được đâu là sự tương đồng về mặt từ loại/đồng nghĩa và đâu là sự liên quan về mặt chủ đề (doctor - hospital). Mọi vector bị kéo lại thành một khối.

2. Đánh đổi giữa Cú pháp và Chủ đề 

Window nhỏ: Mô hình chỉ quan sát các từ sát vách. Nó học được rằng doctor và physician thường đứng ở cùng một vị trí ngữ pháp (ví dụ: sau chữ "the", trước chữ "works"). Do đó, window nhỏ rất xuất sắc trong việc tìm từ đồng nghĩa và nhận diện đặc trưng cú pháp.

Window lớn: Mô hình quan sát bao quát hơn. Nó học được rằng doctor và hospital thường xuyên xuất hiện trong cùng một bối cảnh văn bản, dù chúng hiếm khi đứng cạnh nhau. Do đó, window lớn xuất sắc trong việc nắm bắt chủ đề.

Kết luận rút ra cho sinh viên:
Đừng biến window lớn hay dense vector thành một khẩu hiệu mặc định.

Nếu bạn xây dựng một từ điển từ đồng nghĩa tự động → Hãy chọn window nhỏ.

Nếu bạn làm hệ thống phân loại bài báo y tế và bài báo thể thao → Hãy chọn window lớn.

# 20. Experiment 4 — Embedding dimension

In [6]:
import time
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CHUẨN BỊ DỮ LIỆU (DATA PREPARATION)
# ==========================================
# Tạo một corpus giả lập và nhân bản lên để mô hình có đủ dữ liệu học
base_corpus = [
    ["the", "doctor", "works", "at", "the", "hospital"],
    ["the", "physician", "works", "at", "the", "hospital"],
    ["the", "king", "rules", "the", "kingdom", "as", "a", "man"],
    ["the", "queen", "rules", "the", "kingdom", "as", "a", "woman"],
    ["he", "is", "a", "strong", "man"],
    ["she", "is", "a", "strong", "woman"],
    ["good", "people", "love", "the", "great", "hospital"],
    ["bad", "people", "hate", "the", "terrible", "disease"]
]
# Nhân bản corpus lên 100 lần để mô phỏng dữ liệu lớn hơn
corpus = base_corpus * 100 

# Dữ liệu cho Downstream Task (Phân loại cảm xúc: 1 = Tích cực, 0 = Tiêu cực)
downstream_data = [
    (["love", "great", "good"], 1),
    (["hate", "terrible", "bad", "disease"], 0),
    (["good", "doctor", "hospital"], 1),
    (["bad", "disease"], 0)
]

# Hàm chuyển đổi câu thành vector (bằng cách lấy trung bình cộng vector các từ)
def get_sentence_vector(sentence, w2v_model):
    vectors = [w2v_model.wv[word] for word in sentence if word in w2v_model.wv]
    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)
    return np.mean(vectors, axis=0)

# ==========================================
# 2. TIẾN HÀNH THỰC NGHIỆM (EXPERIMENT)
# ==========================================
dimensions = [50, 100, 300]
results = []

print("Đang chạy thực nghiệm so sánh Dimensions...\n")

for dim in dimensions:
    # 1. Training time: Bắt đầu đo thời gian
    start_time = time.time()
    model = Word2Vec(sentences=corpus, vector_size=dim, window=2, min_count=1, epochs=50, seed=42)
    train_time = time.time() - start_time
    
    # 2. Model size: Đo kích thước của ma trận vector từ vựng (tính bằng Kilobytes)
    model_size_kb = model.wv.vectors.nbytes / 1024
    
    # 3. Similarity: Đo độ tương đồng giữa "doctor" và "physician"
    sim = model.wv.similarity('doctor', 'physician')
    
    # 4. Analogy: King - Man + Woman = ?
    try:
        # Lấy từ đứng đầu bảng xếp hạng
        analogy_result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)[0][0]
    except Exception:
        analogy_result = "N/A"
        
    # 5. Downstream Task: Logistic Regression phân loại cảm xúc
    X = [get_sentence_vector(text, model) for text, label in downstream_data]
    y = [label for text, label in downstream_data]
    
    clf = LogisticRegression()
    clf.fit(X, y) # Huấn luyện classifier
    y_pred = clf.predict(X)
    acc = accuracy_score(y, y_pred) # Đánh giá độ chính xác
    
    # Lưu kết quả
    results.append({
        "Dimension": dim,
        "Train Time (s)": round(train_time, 4),
        "Size (KB)": round(model_size_kb, 2),
        "Similarity (doc-phy)": round(sim, 4),
        "Analogy (K-M+W)": analogy_result,
        "Downstream Acc": acc
    })

# ==========================================
# 3. IN BẢNG KẾT QUẢ SO SÁNH
# ==========================================
df = pd.DataFrame(results)
print(df.to_string(index=False))

Đang chạy thực nghiệm so sánh Dimensions...

 Dimension  Train Time (s)  Size (KB)  Similarity (doc-phy) Analogy (K-M+W)  Downstream Acc
        50          0.0621       5.08                0.9989           queen             1.0
       100          0.0580      10.16                0.9989           queen             1.0
       300          0.0680      30.47                0.9992           queen             1.0


# 22. Evaluation — Word analogy

In [7]:
result = model.wv.most_similar(positive=["king", "woman"], negative=["man"])
print(result[0])

('queen', 0.9945794343948364)


Khi chạy đoạn code trên, kết quả trả về thường là (queen, độ tương đồng). Phép toán vector này hoạt động dựa trên các đặc tính hình học của không gian nhúng, cụ thể như sau:
1. Cơ chế biểu diễn quan hệ bằng hướng và khoảng cách
Các mô hình như Word2Vec học cách biểu diễn từ vựng thông qua ngữ cảnh phân bố của chúng. Quá trình này không chỉ kéo các từ có chung ngữ cảnh lại gần nhau, mà còn mã hóa các mối quan hệ ngữ nghĩa thành các hướng và khoảng cách tương đối đồng nhất trong không gian nhiều chiều.
Phép trừ $\vec{king} - \vec{man}$ sẽ triệt tiêu các giá trị đặc trưng cho nam giới, để lại một vector đại diện cho khái niệm sự trị vì, hoàng gia.
Khi cộng hướng hoàng gia này vào $\vec{woman}$: $\vec{woman} + (\vec{king} - \vec{man})$, hệ thống sẽ tịnh tiến tọa độ của chữ woman trong không gian vector đến vùng giao thoa giữa nữ giới và hoàng gia. Từ vựng có tọa độ gần với điểm đích này nhất chính là queen.
2. Bản chất thống kê và sự hiểu biết
Điểm quan trọng nhất cần ghi nhớ: Sự thành công của các phép toán vector này tuyệt đối không phải là bằng chứng cho thấy mô hình hiểu được ngôn ngữ hay các quan hệ xã hội như con người.
Mô hình không hề biết king là con người, không biết woman là một giới tính, và không có bất kỳ khái niệm nhận thức nào về thế giới thực.
Đây hoàn toàn là một khuôn mẫu thống kê. Trong dữ liệu huấn luyện, cách con người sử dụng từ king và queen có chung một độ lệch phân bố ngữ cảnh giống hệt như cách họ dùng từ man và woman.
Thuật toán chỉ làm một nhiệm vụ duy nhất: tối ưu hóa ma trận để phản ánh lại các xác suất đồng xuất hiện đó. Kết quả vô tình tạo ra các đường thẳng song song trong không gian vector. Do đó, đây là đặc tính của hình học không gian sinh ra từ thống kê, không phải là nhận thức của con người.

# 24. Application — Semantic Search

In [1]:
from gensim.models import Word2Vec

# 1. Tập tài liệu mẫu (Documents)
documents = [
    "the doctor provides clinical therapy for the sick person",
    "a completely unrelated document about football and sports",
    "the patient needs intensive care and therapy"
]

# (Tạo một corpus giả lập bổ sung để mô hình Word2Vec học được các từ khóa y tế)
tokenized_docs = [doc.split() for doc in documents]
extra_corpus = [
    "medical treatment is clinical therapy",
    "medical care for the patient",
    "treatment and therapy for the patient",
    "clinical treatment helps the patient"
]
tokenized_docs.extend([doc.split() for doc in extra_corpus])

# Huấn luyện mô hình Word2Vec
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=50, window=3, min_count=1, epochs=100, sg=1, seed=42)

# 2. Truy vấn ban đầu của người dùng
query_words = ["medical", "treatment"]
expanded_query = set(query_words)

# 3. Mở rộng truy vấn bằng Semantic Similarity
print(f"Truy vấn gốc: {query_words}")
for word in query_words:
    if word in w2v_model.wv:
        # Lấy top 2 từ có ngữ nghĩa gần nhất để mở rộng truy vấn
        similar_words = w2v_model.wv.most_similar(word, topn=2)
        for sim_word, score in similar_words:
            expanded_query.add(sim_word)

print(f"Tập từ khóa sau khi mở rộng (Expanded Query): {expanded_query}\n")

# 4. Tìm kiếm tài liệu dựa trên ngữ nghĩa (Semantic Search)
print("=== KẾT QUẢ TÌM KIẾM ===")
for i, doc in enumerate(documents):
    words_in_doc = set(doc.split())
    # Tính số từ trùng khớp giữa tài liệu và tập truy vấn đã mở rộng
    matched_words = words_in_doc.intersection(expanded_query)
    match_count = len(matched_words)
    
    if match_count > 0:
        print(f"Tài liệu {i+1} (Độ phù hợp: {match_count} từ) - Các từ khớp: {matched_words}")
        print(f"Nội dung: '{doc}'\n")

Truy vấn gốc: ['medical', 'treatment']
Tập từ khóa sau khi mở rộng (Expanded Query): {'treatment', 'needs', 'care', 'medical', 'therapy', 'is'}

=== KẾT QUẢ TÌM KIẾM ===
Tài liệu 1 (Độ phù hợp: 1 từ) - Các từ khớp: {'therapy'}
Nội dung: 'the doctor provides clinical therapy for the sick person'

Tài liệu 3 (Độ phù hợp: 3 từ) - Các từ khớp: {'needs', 'care', 'therapy'}
Nội dung: 'the patient needs intensive care and therapy'

